In [166]:
import os

import ee
import geemap

from IPython.display import Image

In [167]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
try:
    ee.Initialize(project='spatial-yew-490017-r3')
    print("Sucesso: Já estava autenticado!")
except Exception as e:
    print("Iniciando processo de autenticação...")
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='spatial-yew-490017-r3')
    print("Autenticado com sucesso!")
    
    

Sucesso: Já estava autenticado!


In [174]:
# Área de Estudo  - Oeste Baiano
geometry = ee.Geometry.Polygon(
    [
        [
            [-45.85661254590201, -11.587178721087804],
            [-45.85661254590201, -11.825867429555176],
            [-45.56822143262076, -11.825867429555176],
            [-45.56822143262076, -11.587178721087804],
        ]
    ]
)

# ponto central
center = geometry.centroid()

In [175]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.centerObject(center, 11)

In [176]:
# Processamento da Imagem Sentinel-2
collections = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2024-01-01", "2024-09-01")
    .select("B.*")
    .filterBounds(center)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
)

best_img = collections.sort("CLOUDY_PIXEL_PERCENTAGE").first()

print("Qtd imagens:", collections.size().getInfo())
print(
    "Imagem:",
    best_img.get("system:index").getInfo(),
    "| Nuvem:",
    best_img.get("CLOUDY_PIXEL_PERCENTAGE").getInfo(),
)

Qtd imagens: 20
Imagem: 20240702T132239_20240702T132236_T23LMH | Nuvem: 0.000464


In [177]:
Map.addLayer(best_img,
    {"bands": ["B4", "B3", "B2"], "min": 200, "max": 3000, "gamma": 1.2},
    "RGB",
)
Map.addLayer(best_img, {"bands": ["B12", "B8", "B4"], "min": 200, "max": 3000}, "SNR")

Map

Map(center=[-11.706541954880324, -45.712416989261314], controls=(WidgetControl(options=['position', 'transpare…

In [172]:
current_dir = os.getcwd() 
out_gif = 'time_lapse_bahia.gif'
out_gif = os.path.join(current_dir, 'time_lapse_bahia.gif')